# 02 — Regulatory RAG Pipeline

This notebook demonstrates the full Retrieval-Augmented Generation pipeline:
1. Document loading and chunking
2. Embedding with sentence-transformers
3. ChromaDB vector store queries
4. End-to-end Q&A with source citations
5. Retrieval quality analysis

In [ ]:
import sys
sys.path.insert(0, '..')
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
print("Libraries loaded.")

## Step 1 — Document Ingestion & Chunking

In [ ]:
from src.rag.ingestion import load_regulatory_docs, chunk_documents

docs = load_regulatory_docs()
print(f"Documents loaded: {len(docs)}")
for doc in docs:
    print(f"  {doc.metadata['title']} — {len(doc.page_content):,} chars")

chunks = chunk_documents(docs)
print(f"\nTotal chunks: {len(chunks)}")
print(f"Avg chunk size: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} chars")

## Step 2 — Chunk Size Distribution

In [ ]:
chunk_lengths = [len(c.page_content) for c in chunks]
sources = [c.metadata['source'] for c in chunks]

df_chunks = pd.DataFrame({'length': chunk_lengths, 'source': sources})

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(chunk_lengths, bins=20, color='#7c3aed', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Chunk Size (chars)')
axes[0].set_ylabel('Count')
axes[0].set_title('Chunk Size Distribution')
axes[0].axvline(x=500, color='#dc2626', ls='--', label='Target size (500)')
axes[0].legend()

chunk_by_source = df_chunks.groupby('source')['length'].count()
axes[1].barh(chunk_by_source.index, chunk_by_source.values, color='#0f766e', alpha=0.85)
axes[1].set_xlabel('Number of Chunks')
axes[1].set_title('Chunks per Document')

plt.tight_layout()
plt.savefig('../outputs/figures/02_chunk_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 3 — Vector Store Queries

In [ ]:
from src.rag.vectorstore import load_vectorstore

store = load_vectorstore()
print(f"Vector store loaded: {store._collection.count()} vectors")

# Test similarity search
test_queries = [
    "What is the minimum CET1 capital ratio?",
    "When must Enhanced Due Diligence be applied?",
    "What is the Consumer Duty?",
    "How long must AML records be retained?",
]

for query in test_queries:
    results = store.similarity_search(query, k=1)
    top = results[0]
    print(f"\nQuery: {query}")
    print(f"  Source: {top.metadata['title']}")
    print(f"  Chunk:  {top.page_content[:150]}...")

## Step 4 — End-to-End RAG Q&A

In [ ]:
from src.rag.retriever import RegulatoryRAG

rag = RegulatoryRAG()

questions = [
    "What are the minimum capital ratios under Basel III?",
    "What does the FCA Consumer Duty require firms to do?",
    "When must a Suspicious Activity Report be submitted?",
    "What is the Liquidity Coverage Ratio requirement?",
]

for q in questions:
    r = rag.ask(q)
    print(f"Q: {q}")
    print(f"Sources: {', '.join(r.sources)}")
    print(f"Chunks used: {len(r.chunks_used)}")
    print(f"Answer: {r.answer[:200]}...")
    print("-" * 70)

## Step 5 — Retrieval Quality: Semantic Similarity Heatmap

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

queries = [
    "CET1 capital ratio minimum requirement",
    "Enhanced Due Diligence high risk customers",
    "Consumer Duty retail outcomes",
    "Suspicious Activity Report submission",
    "Liquidity Coverage Ratio HQLA",
    "Leverage ratio Tier 1 capital",
]

q_embs = embedder.encode(queries, normalize_embeddings=True)
sim_matrix = q_embs @ q_embs.T

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    sim_matrix,
    xticklabels=[q[:30] + '…' for q in queries],
    yticklabels=[q[:30] + '…' for q in queries],
    annot=True, fmt='.2f', cmap='RdYlGn', center=0.5,
    ax=ax, linewidths=0.5,
)
ax.set_title('Query Semantic Similarity Matrix (cosine)', fontsize=13)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../outputs/figures/02_query_similarity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("Heatmap saved. Low off-diagonal values = queries are semantically distinct (good for RAG diversity).")